# RNN 순환신경망
지금까지 살펴본 신경망은 피드포워드(앞먹임)라는 유형의 신경망으로 흐름이 단방향인 신경망이다. 하지만 이 신경망은 시계열 데이터를 잘 다루지 못한다. 그래서 등장한게 RNN이다.

## 확률과 언어모델
word2vec에서 타겟에 대한 확률은  $P(w_t \mid w_{t-1}, w_{t+1})$ 이여서 맥락을 항상 좌우대칭으로 생각했다 하지만 이번에는 왼쪽 윈도우로 한정해보자. 그 때 확률은  $P(w_t \mid w_{t-2}, w_{t-1})$ 처럼 되고 교차 엔트로피 오차에 의해 유도한 결과는 $L = - \log P(w_t \mid w_{t-2}, w_{t-1})$ 이다. 그래서 CBOW 모델의 학습은 로그 수식의 손실 함수를 최소화 하는 가중치 매개변수를 구하는 것이다. 그럼 이 학습은 무슨 의미가 있을까? 바로 언어 모델이 될 수 있다. 

### 언어모델
언어 모델은 단어의 나열에 확률을 부여하여 특정한 단어의 시퀀스에 대해서, 그 시퀀스가 일어날 가능성이 어느정도 인지를 확률로 평가하는 것이다. 그래서 어떤 문장이 자연스러운지를 평가할 수 있다. 그리고 새로운 문장을 생성하는 용도로도 이용할 수 있다. $$P(w_1, w_2, \dots, w_m) = P(w_1) \cdot P(w_2 | w_1) \cdot P(w_3 | w_1, w_2) \cdot \dots \cdot P(w_m | w_1, \dots, w_{m-1}) = \prod_{t=1}^{m} P(w_t \mid w_1, \dots, w_{t-1})$$ 이는 확률의 곱셈정리로 유도할 수 있다. $$P(A \cap B) = P(A)P(B \mid A)$$ 그래서 정리를 하면 우리의 목표는 $P(w_t \mid w_1, w_2, \dots w_{t-1})$ 를 얻는 것이다.(이를 나타내는 모델을 조건부 언어 모델이라고 한다.) 그리고 저것을 얻을 수 있으면 $P(w_1, \dots, w_{m})$ 도 얻을 수 있다.

### CBOW 모델을 언어모델로?
억지로 언어모델에 적용하려면 어떻게 해야할까? 맥락의 크기를 특정 값으로 한정 및 고정해 근사적으로 나타낼 수 있다. $$P(w_1, w_2, \dots, w_m) = \prod_{t=1}^{m} P(w_t \mid w_1, \dots, w_{t-1})  \approx P(w_1, \dots, w_m) \approx \prod_{t=1}^{m} P(w_t \mid w_{t-2}, w_{t-1})$$ 하지만 맥락이 2개로는 일반적으로 부족하다 문장이 두 단어로 이어지지 않는 경우도 있기 때문이다. 그렇다고 맥락의 크기를 늘린다고 해결될까? 하지만 CBOW모델에서는 맥락 안의 단어 순서가 무시된다. 그래서 맥락의 단어 벡터를 은닉층에서 연결하는 방식을 생각할 수 있다. 하지만 이 방식에서는 맥락의 크기에 비례해 가중치 매개변수가 늘어나게 된다. 그래서 우리는 RNN을 고안하였다. 

#### Note
머신러닝과 통계학에서는 마르코프 연쇄, 마르코프 모델이라는 말을 자주 하는데 마르코프 연쇄란 미래의 상태가 현재 상태에만 의존해 결정되는 것을 말한다. 또한 이 사상의 확률이 그 직전 N개의 사건에만 의존할 때 이를 N층 마르코프 연쇄라고 한다. 이번 예는 직전 2개의 단어에만 의존하여 다음 단어가 정해지는 모델이므로 2층 마르코프 연쇄라고 할 수 있다.

## RNN
순환한다는 반복해서 되돌아간다는 뜻으로 닫힌 경로가 필요함을 내포한다. 그래서 이 닫힌 경로를 따라 데이터는 끊임 없이 순환할 수 있다. 그래서 과거의 정보를 기억하고 최신 데이터를 갱신할 수 있다.   
시계열 데이터가 RNN 계층의 입력이 되고 각 시각의 RNN 계층은 그 계층으로의 입력과 1개 전의 RNN계층으로부터의 출력을 입력으로 받는다. 그리고 이 두 정보를 바탕으로 현 시각의 출력을 계산한다. 그래서 이 수식을 만족한다. $$h_{t} = tanh(h_{t-1}W_{h} + x{t}W_{x} + b)$$ 여기서 입력 x를 출력 h로 변화가히 위한 가중치 $W_{x}$이고 다른 하나는 다음 시각의 출력으로 변환하기 위한 가중치 $W_{h}$이다. 그리고 편향 b도 있고 $h_{t-1}, x_{t}$은 행 백터이다.  
그래서 행렬 곱을 계산한 뒤에 tanh함수를 이용해서 변환되고 그 결과는 출력 $h_{t}$가 된다. 그리고 이 출력은 다른 계층을 향해 위쪽으로 출력되는 동시에 다음 시각의 RNN계층을 향해 출력된다.   
그리고 수식을 잘 보면 현재 출력은 한 시각 이전 출력에 기초해 계산 되므로 RNN은 상태를 가지고 있고 갱신된다고 볼 수 있다.

### BPTT
순환 구조를 펼친 후 RNN은 오차역전파법을 적용할 수 있다. 여기서 오차역전파법은 시간방향으로 펼친 신경망의 오차역전파법이라는 뜻으로 BPTT(Backpropagation Through Time)라고 한다. 이렇게 보면 학습시킬 수 있어 보이지만 긴 시계열 데이터를 학습할 때 문제가 시간의 크기가 커지는 것에 비례하여 BPTT가 소비하는 컴퓨팅 자원도 증가하기 때문이다. 그래서 역전파의 기울기가 불안정해진다.

### Truncated BPTT 
그래서 큰 시계열 데이터를 취급할 때 신경망 연결을 적당한 길이로 잘라내어 작은 신경망에서 오차역전파법을 수행하는데 이를 Truncated BPTT 기법이라 한다. 그런데 이를 제대로 구현하려면 역전파의 연결만 끊어야지 순전파는 유지되어야 한다.  

### Truncated BPTT 미니배치 학습
우리는 미니배치 학습을 수행하므로 데이터를 입력할 때 순서대로 입력해야하고 이렇게 하려면 데이터를 주는 시작 위치를 각 미니배치의 시작 위치로 옮겨줘야 한다. 